In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
parent_llm = ChatOpenAI()
subGraph_llm = ChatOpenAI()

In [3]:
# Parent graph
class ParentGraphState(TypedDict):
    question: str
    answer_eng: str
    answer_hindi: str

In [21]:
def generate_answer(state: ParentGraphState):
    question = state["question"]
    response = parent_llm.invoke(f"Answer the following question in English: {question}")
    return {"answer_eng": response.content}

def translate_answer(state: ParentGraphState):
    answer_eng = state["answer_eng"]
    prompt = f"Translate the following text to Hindi: {answer_eng}"
    subgraph_res = subGraph_llm.invoke(prompt)
    return {"answer_hindi": subgraph_res.content}

In [24]:
# pass shared state to subgraph
subgraph = StateGraph(ParentGraphState)

subgraph.add_node("translate", translate_answer)

subgraph.add_edge(START, "translate")
subgraph.add_edge("translate", END)

subgraph_agent = subgraph.compile()

In [23]:
parent_graph = StateGraph(ParentGraphState)

parent_graph.add_node("generate", generate_answer)
# using subgraph as node
parent_graph.add_node("translate", subgraph_agent)

parent_graph.add_edge(START, "generate")
parent_graph.add_edge("generate", "translate")
parent_graph.add_edge("translate", END)

parent_agent = parent_graph.compile()

# Example usage
initial_state = {"question": "What is the quantum mechanical model?"}
final_state = parent_agent.invoke(initial_state)
print(final_state)

{'question': 'What is the quantum mechanical model?', 'answer_eng': 'The quantum mechanical model is a modern understanding of the behavior of atoms and subatomic particles based on quantum mechanics. It describes the probability distribution of electrons in an atom, rather than their specific paths, and incorporates the principles of wave-particle duality and uncertainty. This model has replaced the older Bohr model of the atom and has provided a more accurate description of atomic behavior.', 'answer_hindi': 'क्वांटम मैकेनिकल मॉडल एक आधुनिक समझ है जो क्वांटम मैकेनिक्स पर आधारित अणुओं और अणु-कणों के व्यवहार की। यह इलेक्ट्रॉनों की संभावना वितरण का वर्णन व्यक्तिगत पथों की बजाय करता है, और तरंग-कण द्वित्व और अनिश्चितता के सिद्धांतों को सम्मिलित करता है। यह मॉडल पुराने बोहर मॉडल को बदल दिया है और अणु के व्यवहार का अधिक सटीक वर्णन प्रदान किया है।'}
